# Task 3: Heart Disease Prediction
**Internship:** DevelopersHub Corporation – AI/ML Engineering

**Objective:** Build a model to predict whether a person is at risk of heart disease based on health data.

**Dataset:** Heart Disease UCI Dataset (loaded from a public URL – no manual download needed!)

## Step 1: Import Libraries

In [ ]:
# All required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, roc_auc_score
)

print("Libraries imported successfully!")

## Step 2: Load the Dataset

In [ ]:
# ============================================================
# DATASET SETUP — READ THIS FIRST (only takes 1 minute)
# ============================================================
# The Heart Disease UCI dataset must be downloaded from Kaggle.
#
# STEP 1: Go to this link and click Download:
#   https://www.kaggle.com/datasets/ronitf/heart-disease-uci
#   (you need a free Kaggle account)
#
# STEP 2: You will get a file called "heart.csv"
#
# STEP 3: Upload it to this Colab session:
#   - Click the folder icon on the LEFT sidebar of Colab
#   - Click the Upload button (paper with arrow icon)
#   - Select heart.csv from your Downloads folder
#
# STEP 4: Run this cell — it will load automatically!
# ============================================================

import os

if os.path.exists("heart.csv"):
    df = pd.read_csv("heart.csv")
    print("heart.csv loaded successfully!")
else:
    raise FileNotFoundError(
        "heart.csv not found!\n\n"
        "Please follow these steps:\n"
        "1. Download from: https://www.kaggle.com/datasets/ronitf/heart-disease-uci\n"
        "2. Click the folder icon in the LEFT sidebar of Colab\n"
        "3. Click Upload and select heart.csv\n"
        "4. Run this cell again"
    )

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


## Step 3: Understand the Features

In [ ]:
# What do the columns mean?
feature_info = {
    'age':      'Age of the patient',
    'sex':      '1 = Male, 0 = Female',
    'cp':       'Chest pain type (0-3)',
    'trestbps': 'Resting blood pressure (mmHg)',
    'chol':     'Cholesterol level (mg/dl)',
    'fbs':      'Fasting blood sugar > 120 mg/dl (1=True)',
    'restecg':  'Resting ECG results (0-2)',
    'thalach':  'Max heart rate achieved',
    'exang':    'Exercise induced angina (1=Yes)',
    'oldpeak':  'ST depression induced by exercise',
    'slope':    'Slope of peak exercise ST segment',
    'ca':       'Number of major vessels (0-3)',
    'thal':     'Thalassemia (3=normal, 6=fixed defect, 7=reversible)',
    'target':   'Heart Disease: 1 = Yes, 0 = No (our prediction goal)'
}

for col, desc in feature_info.items():
    print(f"  {col:12s}: {desc}")

## Step 4: Data Cleaning

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Check data types
print("\nData types:")
print(df.dtypes)

# Check target class distribution
print("\nTarget distribution:")
print(df['target'].value_counts())
print(f"  Heart Disease cases: {df['target'].sum()} ({df['target'].mean()*100:.1f}%)")

## Step 5: Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of heart disease vs no disease
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='Set1')
plt.title('Heart Disease Distribution (0=No, 1=Yes)')
plt.xlabel('Heart Disease')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Age vs Max Heart Rate colored by disease status
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='age', y='thalach', hue='target',
                palette={0:'steelblue', 1:'tomato'}, alpha=0.7)
plt.title('Age vs Max Heart Rate (Red = Has Heart Disease)')
plt.xlabel('Age')
plt.ylabel('Max Heart Rate Achieved')
plt.legend(title='Heart Disease')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='RdYlGn', linewidths=0.4)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Chest pain type vs heart disease
plt.figure(figsize=(7, 4))
sns.countplot(x='cp', hue='target', data=df, palette='Set1')
plt.title('Chest Pain Type vs Heart Disease')
plt.xlabel('Chest Pain Type (0-3)')
plt.ylabel('Count')
plt.legend(title='Heart Disease', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

## Step 6: Preprocessing – Split and Scale

In [ ]:
# Features and target
X = df.drop('target', axis=1)
y = df['target']

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

## Step 7: Train Models

In [ ]:
# --- Model 1: Logistic Regression ---
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_preds = lr.predict(X_test_scaled)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]

lr_acc = accuracy_score(y_test, lr_preds)
lr_auc = roc_auc_score(y_test, lr_probs)

print(f"Logistic Regression — Accuracy: {lr_acc:.4f} | AUC: {lr_auc:.4f}")

# --- Model 2: Decision Tree ---
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)
dt_probs = dt.predict_proba(X_test)[:, 1]

dt_acc = accuracy_score(y_test, dt_preds)
dt_auc = roc_auc_score(y_test, dt_probs)

print(f"Decision Tree      — Accuracy: {dt_acc:.4f} | AUC: {dt_auc:.4f}")

## Step 8: Confusion Matrix

In [ ]:
# Plot confusion matrices for both models side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in zip(
    axes,
    [lr_preds, dt_preds],
    ['Logistic Regression', 'Decision Tree']
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'], ax=ax)
    ax.set_title(f'{title} – Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Step 9: ROC Curve

In [ ]:
# ROC Curves for both models
plt.figure(figsize=(8, 5))

for probs, name, auc_val in [
    (lr_probs, 'Logistic Regression', lr_auc),
    (dt_probs, 'Decision Tree',       dt_auc)
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})', lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – Heart Disease Prediction')
plt.legend()
plt.tight_layout()
plt.show()

## Step 10: Feature Importance

In [ ]:
# Feature importance from Decision Tree
importances = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette='Reds_r')
plt.title('Feature Importance – Decision Tree')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("Top features affecting heart disease prediction:")
print(importances)

## Step 11: Classification Report

In [ ]:
print("Logistic Regression – Detailed Report:")
print(classification_report(y_test, lr_preds, target_names=['No Disease', 'Heart Disease']))

## Step 12: Key Insights and Conclusion

In [ ]:
print(f"""
KEY INSIGHTS – HEART DISEASE PREDICTION:
==========================================
1. Dataset: 303 patients, 13 health features, binary target (disease or not).
2. No missing values were found – dataset was clean.
3. Logistic Regression Accuracy: {lr_acc*100:.1f}% | AUC: {lr_auc:.3f}
4. Decision Tree Accuracy:       {dt_acc*100:.1f}% | AUC: {dt_auc:.3f}
5. Most important features: chest pain type (cp), thalach (max heart rate),
   thal (thalassemia), and oldpeak (ST depression).
6. Higher max heart rate and certain chest pain types correlate strongly
   with heart disease presence.
7. Logistic Regression is more reliable for medical predictions due to
   better generalization and probability outputs.
""")